In [1]:
import os 
from flask import Flask,render_template,request,redirect,jsonify,flash,url_for,make_response
from flask_sqlalchemy import SQLAlchemy
import sqlalchemy
from sqlalchemy import create_engine,Table,Column,Integer,String,ForeignKey
from sqlalchemy.orm import Session
import time
from datetime import datetime,date
import json
from flask_login import LoginManager,login_user,logout_user,current_user,login_required,UserMixin
from flask_restful import Resource,reqparse,Api
from flask import current_app as app
from flask_cors import CORS
from werkzeug.exceptions import HTTPException
import re
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
app=Flask(__name__)
CORS(app)
app.config['SQLALCHEMY_DATABASE_URI']= "sqlite:///"+os.path.join(os.getcwd(),'database.sqlite3')
db=SQLAlchemy()
db.init_app(app)
api=Api(app)
app.app_context().push()
login_manager = LoginManager()
login_manager.init_app(app)
#app.secret_key = b'_5#y2L"F4Q8z\n\xec]/'
app.secret_key = b'ajiuhaeiu"4%^&FGG??'
class NotfoundError(HTTPException):
    def __init__(self,status_code):
        self.response=make_response("",status_code)

        
class AlreadyExists(HTTPException):
    def __init__(self,status_code,code):
        self.response=make_response(f"{code} already exist",status_code)

        
class BusinessValidationError(HTTPException):
    def __init__(self,status_code,error_code,error_message):
        message={"error_code":error_code,"error_message":error_message}
        self.response=make_response(jsonify(message),status_code)
class Movie(db.Model):
    __tablename__ = 'movie'
    id = Column(Integer, autoincrement=True, primary_key=True)
    name = Column(String,nullable=False)
    caption = Column(String,nullable=False)
    rating = Column(Integer,nullable=False)
    tags = Column(String,nullable=False)
    no_of_rating = Column(Integer,nullable=False)
class User(db.Model,UserMixin):
    __tablename__ = 'user'
    id = Column(Integer, autoincrement=True, primary_key=True)
    name = Column(String,nullable=False)
    age = Column(Integer,nullable=False)
    gender = Column(String,nullable=False)
    username = Column(String,unique=True, nullable=False)
    password=Column(String,nullable=False)
    auth=Column(Integer,nullable=False,default=0)
    @property
    def is_authenticated(self):        
        return self.auth==1
    is_active=True
    is_anonymous=False
    def get_id(self):
        return str(self.id)


class User_Movie(db.Model):
    __tablename__ = 'user_movie'
    id = Column(Integer, autoincrement=True, primary_key=True)
    movie_id = Column(Integer,nullable=False)
    user_id = Column(Integer,nullable=False)
    like = Column(Integer,nullable=False)
    cart = Column(Integer,nullable=False)

In [14]:
movie_tags = {}
for i in Movie.query.all():
    st = f" {i.tags}"
    pattern = r'\b[a-zA-Z]+\b|\b\d{4}\b'
    tl = re.findall(pattern, st)
    tl = [i.lower() for i in tl ] 
    movie_tags[i.name] = tl

tags = sorted(set(tag for tags_list in movie_tags.values() for tag in tags_list))
user_likes = ['Doctor Strange','Cars 3']
if len(user_likes) == 0:
    user_likes = list(movie_tags.keys())
tag_matrix = np.zeros((len(movie_tags), len(tags)))
for i, movie_id in enumerate(movie_tags.keys()):
    for tag in movie_tags[movie_id]:
        j = tags.index(tag)
        tag_matrix[i, j] = 1
liked_indices = [list(movie_tags.keys()).index(movie_id) for movie_id in user_likes]
liked_vector = tag_matrix[liked_indices]
liked_vector = liked_vector.reshape(1, -1)
similarity_scores = cosine_similarity(tag_matrix, liked_vector.reshape(len(user_likes),len(tags)))
similarity_scores = similarity_scores.sum(axis=1)
ml = list(movie_tags.keys())
ml = [(ml[i],similarity_scores[i]) for i in range(len(ml))]
sorted(ml,key = lambda x : -x[1])

[('Finding Dory', 1.5188296772376009),
 ('Captain America: Civil War', 1.5039526306789694),
 ('Doctor Strange', 1.5039526306789694),
 ('Cars 3', 1.5039526306789694),
 ('Thor: Ragnarok', 1.4870836454915688),
 ('Pirates of the Caribbean - Salazar s Revenge', 1.4702146603041686),
 ('Guardians Of The Galaxy Vol. 2', 1.4519305672502312),
 ('The Jungle Book', 1.4077185661264897),
 ('Pete s Dragon', 1.4077185661264897),
 ('Luca', 1.3928415195678583),
 ('The Good Dinosaur', 1.3928415195678583),
 ('Raya and the Last Dragon', 1.3928415195678583),
 ('Cars', 1.3928415195678583),
 ('Finding Nemo', 1.3928415195678583),
 ('Toy Story', 1.3928415195678583),
 ('A Bug s Life', 1.3928415195678583),
 ('Brave', 1.3928415195678583),
 ('Toy Story 4', 1.3928415195678583),
 ('Dinosaur', 1.3928415195678583),
 ('Bolt', 1.3928415195678583),
 ('Toy Story 3', 1.3928415195678583),
 ('Toy Story 2', 1.3928415195678583),
 ('Monsters, Inc.', 1.3928415195678583),
 ('Snow White And The Seven Dwarfs', 1.3928415195678583),
 